In [1]:
import os
import cv2
import shutil
import numpy as np
from tqdm import tqdm

# Configuration
MASK_DIR = "cleaned_dataset/mask_rotated"
FAIL_DIR = "cleaned_dataset/failed_masks"
os.makedirs(FAIL_DIR, exist_ok=True)

def get_mask_status(file_path):
    # Load as grayscale
    mask = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
    if mask is None: return "corrupt"

    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return "empty"

    # Get the largest contour
    cnt = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(cnt)
    
    # 1. Pixel Count (Actual white pixels)
    white_pixels = cv2.countNonZero(mask)

    # 2. Geometric measurements
    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull)
    x, y, w, h = cv2.boundingRect(cnt)
    bbox_area = w * h

    # --- THE AGGRESSIVE FILTERS ---

    # A. Density Filter (THE BIG ONE)
    # A solid egg mask's white pixels should fill > 95% of its convex hull.
    # The grid-noise masks usually hit 40-70% because of the black gaps.
    density = white_pixels / hull_area if hull_area > 0 else 0
    if density < 0.90:  # Adjust to 0.95 if still seeing holes
        return f"low_density_{density:.2f}"

    # B. Solidity Filter
    # Detects if the outer shape is "lumpy" or fragmented
    solidity = area / hull_area if hull_area > 0 else 0
    if solidity < 0.94: 
        return f"low_solidity_{solidity:.2f}"

    # C. Bounding Box Fill
    # An oval should take up about 78% of its bounding box (pi/4).
    # If it's way lower, it's likely a thin diagonal line or scattered noise.
    bbox_fill = area / bbox_area if bbox_area > 0 else 0
    if bbox_fill < 0.60:
        return f"low_bbox_fill_{bbox_fill:.2f}"

    # D. Aspect Ratio
    aspect_ratio = float(w) / h if h > 0 else 0
    if aspect_ratio > 1.8 or aspect_ratio < 0.55:
        return "wrong_proportions"

    return "pass"

# Process
all_files = [f for f in os.listdir(MASK_DIR) if f.lower().endswith(('.jpg', '.png'))]
print(f"🚀 Running Aggressive Filter on {len(all_files)} files...")

stats = {"pass": 0, "fail": 0}
for filename in tqdm(all_files):
    path = os.path.join(MASK_DIR, filename)
    status = get_mask_status(path)
    
    if status != "pass":
        shutil.move(path, os.path.join(FAIL_DIR, filename))
        stats["fail"] += 1
    else:
        stats["pass"] += 1

print(f"\nDone! Cleaned out {stats['fail']} noisy masks. {stats['pass']} remaining.")

🚀 Running Aggressive Filter on 4588 files...


100%|██████████| 4588/4588 [00:02<00:00, 1616.43it/s]


Done! Cleaned out 0 noisy masks. 4588 remaining.


In [3]:
import os
import pandas as pd
from pathlib import Path

# Path Configuration
CSV_PATH = "cleaned_dataset/egg_metadata.csv"
MASK_DIR = "cleaned_dataset/mask_rotated"
OUTPUT_CSV = Path("cleaned_dataset/new_csv/filtered_metadata.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# 1. Load the original metadata
df = pd.read_csv(CSV_PATH)

# 2. Get the list of filenames currently in the filtered mask folder
# We use a set for O(1) lookup speed
filtered_masks = set(os.listdir(MASK_DIR))

# 3. Filter the dataframe
# Using 'image_path' as the key based on your CSV snippet
# Adjust 'image_path' to 'mask_filename' if your mask folder uses a different naming convention
if 'image_path' in df.columns:
    initial_count = len(df)
    filtered_df = df[df['image_path'].isin(filtered_masks)]
    final_count = len(filtered_df)
    
    # 4. Save the new metadata
    filtered_df.to_csv(OUTPUT_CSV, index=False)
    
    print(f"📊 Filtering Summary:")
    print(f"--- Original rows: {initial_count}")
    print(f"--- Survived rows: {final_count}")
    print(f"--- Removed: {initial_count - final_count} noisy cases")
    print(f"✅ Clean metadata saved to: {OUTPUT_CSV}")
else:
    print(f"❌ Error: 'image_path' column not found in CSV. Available columns: {df.columns.tolist()}")

📊 Filtering Summary:
--- Original rows: 4633
--- Survived rows: 4588
--- Removed: 45 noisy cases
✅ Clean metadata saved to: cleaned_dataset\new_csv\filtered_metadata.csv
